# TXC Library Verification Notebook

This notebook demonstrates that the new `m02_tx_integrator` package runs correctly in a notebook, produces expected numerical outputs, and that the plotting helpers render as expected.

The structure follows a critical validation workflow: smoke tests, numerical checks, plotting checks, failure injection, and a devil's advocate debug pass.

## 1. Install and Import Dependencies

The package is already present in the workspace. This cell imports the local library directly from the editable workspace path so the notebook does not depend on external packaging state.

In [ ]:
from pathlib import Path
import sys

workspace_root = Path.cwd()
package_src = workspace_root / "m02_tx_integrator" / "src"
if str(package_src) not in sys.path:
    sys.path.insert(0, str(package_src))

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

from m02_tx_integrator import Exporter, Plotter, TXCIntegrator, nevada_sand_original, reference_tests

## 2. Verify Library Versions and Runtime Environment

This cell surfaces runtime details that often explain notebook-only failures: Python path resolution, matplotlib backend, and package versions.

In [ ]:
print(f"Python executable: {sys.executable}")
print(f"Workspace root: {workspace_root}")
print(f"Package source exists: {package_src.exists()}")
print(f"matplotlib backend: {matplotlib.get_backend()}")
print(f"pandas version: {pd.__version__}")

## 3. Run a Minimal Integration Smoke Test

For this project, the relevant smoke test is a single TXC material-point run using one of the original reference cases.

In [ ]:
material, stiffness = nevada_sand_original()
single_test = reference_tests()[0]
integrator = TXCIntegrator(material, stiffness)
single_result = integrator.run(single_test)

print(f"Test label: {single_result.label}")
print(f"Points: {len(single_result.p)}")
print(f"ppeak = {single_result.ppeak:.8f}")
print(f"qpeak = {single_result.qpeak:.8f}")
print(f"pcs   = {single_result.pcs:.8f}")
print(f"qcs   = {single_result.qcs:.8f}")

single_result.to_dataframe().head()

## 4. Validate Numerical Results Against Expected Values

The expected values below were captured from the same library after an earlier smoke validation. This is not a substitute for full regression data from the legacy script, but it is a strong notebook-level consistency check.

In [ ]:
expected = {
    "ppeak": 79.45921091,
    "qpeak": 120.62040673,
    "pcs": 70.06071929,
    "qcs": 90.18215787,
}
tolerance = 1e-8

checks = []
for key, expected_value in expected.items():
    actual_value = getattr(single_result, key)
    abs_error = abs(actual_value - expected_value)
    rel_error = abs_error / max(abs(expected_value), 1.0)
    checks.append({
        "quantity": key,
        "actual": actual_value,
        "expected": expected_value,
        "abs_error": abs_error,
        "rel_error": rel_error,
        "passes": abs_error < tolerance,
    })

checks_df = pd.DataFrame(checks)
display(checks_df)
assert checks_df["passes"].all(), "Single-test regression check failed"

## 5. Render a Minimal Plotter Smoke Test

The first check is simply that the plotting API returns live matplotlib axes and renders without backend-specific errors.

In [ ]:
axis = Plotter.plot_stress_strain([single_result])
axis.set_title("Single-test stress-strain smoke test")
plt.show()

assert axis.get_xlabel() == "eax (%)"
assert axis.get_ylabel() == "q"

## 6. Plot Integrated Results Across Multiple Inputs

This checks the plotter against real batch output rather than only a single curve.

In [ ]:
batch_tests = reference_tests()[:3]
batch_results = integrator.run_batch(batch_tests)

summary_df = pd.DataFrame(
    {
        "label": [result.label for result in batch_results],
        "final_p": [result.p[-1] for result in batch_results],
        "final_q": [result.q[-1] for result in batch_results],
        "peak_q": [result.qpeak for result in batch_results],
    }
)
display(summary_df)

Plotter.plot_stress_path(batch_results)
plt.title("Stress paths for three reference tests")
plt.show()

Plotter.plot_volumetric(batch_results)
plt.title("Volumetric response for three reference tests")
plt.show()

## 7. Add Assertions for Numerical and Plot Output Checks

Assertions turn the notebook into a lightweight executable test rather than just a visual demo.

In [ ]:
assert len(single_result.eax) == 2001
assert len(single_result.q) == 2001
assert len(single_result.gtan) == 2001
assert checks_df["abs_error"].max() < tolerance
assert summary_df["peak_q"].notna().all()
assert (summary_df["peak_q"] > 0.0).all()

export_dir = workspace_root / "m02_tx_integrator" / "notebook_demo_output"
written_files = Exporter.to_csv_batch(batch_results, export_dir)
print([path.name for path in written_files])
assert len(written_files) == len(batch_results)

## 8. Trigger Common Failure Cases

A useful notebook should show failures on purpose, then explain why they are failures instead of silently masking them.

In [ ]:
failure_messages = []

try:
    integrator.run(single_test, n_elastic_steps=0)
except Exception as exc:
    failure_messages.append(f"Zero elastic steps: {type(exc).__name__}: {exc}")

try:
    broken_material, broken_stiffness = nevada_sand_original()
    TXCIntegrator(broken_material, broken_stiffness).run(
        single_test,
        peak_tolerance=1e-20,
        max_peak_iterations=1,
    )
except Exception as exc:
    failure_messages.append(f"Peak solver under-iterated: {type(exc).__name__}: {exc}")

for message in failure_messages:
    print(message)

assert len(failure_messages) == 2

## 9. Debug with a Critical Review Checklist

Devil's advocate review asks whether the failure comes from the code, the notebook state, or a bad assumption. Use this checklist before changing implementation details:

- Are we importing the intended local package path rather than a stale installed copy?
- Are step counts still at the validated defaults of 1000 and 1000?
- Did the peak solver fail because convergence is genuinely poor, or because the requested tolerance and iteration limit were unrealistic?
- Are plots missing because `plt.show()` was skipped or because the backend is non-interactive?
- Are repeated notebook runs accumulating stale figures or overwritten variables?
- Are we comparing against a compatible reference output schema?

## 10. Apply Fixes and Re-run Validation

The failure cases above were deliberate. The fixes are to restore validated defaults and reasonable solver settings, then rerun the trusted path.

In [ ]:
validated_result = integrator.run(single_test, peak_tolerance=1e-10, max_peak_iterations=100)
validated_axis = Plotter.plot_stress_path([validated_result])
validated_axis.set_title("Validated stress path after restoring defaults")
plt.show()

assert len(validated_result.p) == 2001
assert abs(validated_result.ppeak - expected["ppeak"]) < tolerance
assert abs(validated_result.qpeak - expected["qpeak"]) < tolerance
print("Validation rerun completed successfully.")

## Conclusion

The notebook confirms that the library runs, batch execution works, CSV export works, and the plotter returns working matplotlib axes. The deliberate failure cases also show that the library surfaces invalid configuration and solver under-iteration clearly instead of failing silently.